In [3]:

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

# Get the absolute path to the directory containing the current script
# For GlobalLocal/src/analysis/preproc/make_epoched_data.py, this is GlobalLocal/src/analysis/preproc
# Get the absolute path to the directory containing the current script
try:
    # This will work if running as a .py script
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    # This will be executed if __file__ is not defined (e.g., in a Jupyter Notebook)
    current_script_dir = os.getcwd()

# Navigate up two levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..', '..'))

# Add the 'GlobalLocal' directory to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # insert at the beginning to prioritize it

from ieeg.navigate import channel_outlier_marker, trial_ieeg, crop_empty_data, \
    outliers_to_nan
from ieeg.io import raw_from_layout, get_data
from ieeg.timefreq.utils import crop_pad
from ieeg.timefreq import gamma
from ieeg.calc.scaling import rescale
import mne
import numpy as np
from ieeg.calc.stats import time_perm_cluster, window_averaged_shuffle
from ieeg.viz.mri import gen_labels
import logging

import matplotlib.pyplot as plt
from collections import OrderedDict, defaultdict
import json
# still need to test if the permutation test functions load in properly.
import pandas as pd
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# from src.analysis.power.roi_analysis import blah_blah
from src.analysis.config import experiment_conditions
from src.analysis.config.plotting_parameters import plotting_parameters as plot_params
import src.analysis.utils.general_utils as utils # import utils functions one by one by name
from src.analysis.power.power_traces import make_multi_channel_evokeds_for_all_conditions_and_rois, \
                                            plot_power_traces_for_all_rois, create_subtracted_evokeds_dict, \
                                            time_perm_cluster_between_two_evokeds, process_windowed_data_for_anova, \
                                            create_windowed_anova_dataframe, perform_windowed_anova, \
                                            apply_fdr_correction_to_windowed_results
from src.analysis.config import experiment_conditions


In [11]:
# hardcoding lab root (Erin)
LAB_root = "/cwork/jz421"
# EPOCHS_ROOT_FILE = "Stimulus_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"
EPOCHS_ROOT_FILE = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"
subjects = ['D0057']
config_dir = os.path.join(project_root, 'src', 'analysis', 'config')
subjects_electrodestoROIs_dict = utils.load_subjects_electrodes_to_ROIs_dict(save_dir=config_dir, filename='subjects_electrodestoROIs_dict.json')
conditions = experiment_conditions.stimulus_err_corr_conditions # set this to whichever conditions you're running


# Get data layout
layout = get_data("GlobalLocal", root=LAB_root)

subjects_mne_objects = utils.create_subjects_mne_objects_dict(subjects, epochs_root_file=EPOCHS_ROOT_FILE, conditions=experiment_conditions.stimulus_err_corr_conditions, task="GlobalLocal", just_HG_ev1_rescaled=True, acc_trials_only=False)

Loaded data from /hpc/group/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json
Loading data for subject: D0057
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0057/D0057_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation matrices available
Adding metadata with 29 columns
449 matching events found
No baseline correction applied
0 projection items activated
Replacing existing metadata with 29 columns
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0057/D0057_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_power_rescaled-

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:449: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_err: 32 valid trials out of 32
  Loading condition: Stimulus_corr


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:449: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


Adding metadata with 29 columns
405 matching events found
No baseline correction applied
    Stimulus_corr: 405 valid trials out of 405
  Loading condition: Stimulus_err
Adding metadata with 29 columns
32 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:449: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_err: 32 valid trials out of 32
  Loading condition: Stimulus_corr


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:449: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


Adding metadata with 29 columns
405 matching events found
No baseline correction applied
    Stimulus_corr: 405 valid trials out of 405


In [12]:
subjects_mne_objects['D0057']['Stimulus_err']['HG_ev1_rescaled'].get_data().shape

(32, 175, 641)